# Cross-Dataset RAC Evaluation

Train on ISHate or Vicomtech, always evaluate on IHC test set. Uses the same RAC pipeline as `training.ipynb`.

## 1. Imports

In [1]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import faiss
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import load_dataset, Dataset
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
import sys
sys.path.insert(0, str(Path("..").resolve()))
from retriever import encode, retrieve_top_k_above_threshold

## 2. Configuration

Here you can choose which models and index types to run. Modify `SELECTED_MODELS`, `SELECTED_INDEX_TYPES`, `TRAIN_DATASETS` and `EVAL_DATASET` as you like.

In [2]:
WEIGHTS_RAC_DIR = Path('../..') / 'weigths' / 'weights_rac_cross_evaluation'
INDEX_DIR       = Path('../..') / 'corpus' / 'index'
CHUNKS_DIR      = Path('../..') / 'corpus' / 'chunks'

MODELS = {
    'bert':     'bert-base-uncased',
    'hatebert': 'GroNLP/hateBERT',
    'roberta':  'roberta-base',
}

RETRIEVER_HF_ID = 'sentence-transformers/all-mpnet-base-v2'

# === Cross-dataset evaluation: train on TRAIN_DATASETS, always test on EVAL_DATASET ===
SELECTED_MODELS      = ['bert', 'roberta']
SELECTED_INDEX_TYPES = ['example', 'knowledge', 'full']
TRAIN_DATASETS       = ['ISHate', 'Vicomtech']
EVAL_DATASET         = 'IHC'

# Retrieval config
K         = 5
THRESHOLD = 0.3

# Training config
MAX_LENGTH    = 256
BATCH_SIZE    = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS    = 3

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device            : {device}')
print(f'Retriever         : {RETRIEVER_HF_ID}')
print(f'k / threshold     : {K} / {THRESHOLD}')
print(f'Models            : {SELECTED_MODELS}')
print(f'Index types       : {SELECTED_INDEX_TYPES}')
print(f'Train datasets    : {TRAIN_DATASETS}')
print(f'Eval dataset      : {EVAL_DATASET}')

Device            : cuda
Retriever         : sentence-transformers/all-mpnet-base-v2
k / threshold     : 5 / 0.3
Models            : ['bert', 'roberta']
Index types       : ['example', 'knowledge', 'full']
Train datasets    : ['ISHate', 'Vicomtech']
Eval dataset      : IHC


## 3. Load Datasets

Load all three datasets using `data_loaders.py`.

In [3]:
from data_loaders import load_ihc_binary, load_ishate_binary, load_vicomtech

train_ihc, test_ihc     = load_ihc_binary(seed=42)
train_ishate, test_ishate = load_ishate_binary()
vicomtech_train, vicomtech_test = load_vicomtech()

DATASETS = {
    'IHC':       {'train': train_ihc,       'test': test_ihc,       'text_col': 'post'},
    'ISHate':    {'train': train_ishate,     'test': test_ishate,    'text_col': 'text'},
    'Vicomtech': {'train': vicomtech_train,  'test': vicomtech_test, 'text_col': 'text'},
}

print(f'IHC        — train: {len(train_ihc):,}  test: {len(test_ihc):,}')
print(f'ISHate     — train: {len(train_ishate):,}  test: {len(test_ishate):,}')
print(f'Vicomtech  — train: {len(vicomtech_train):,}  test: {len(vicomtech_test):,}')

README.md:   0%|          | 0.00/792 [00:00<?, ?B/s]

implicit_hate_v1_stg1_posts.tsv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/21480 [00:00<?, ? examples/s]

Map:   0%|          | 0/19332 [00:00<?, ? examples/s]

Map:   0%|          | 0/2148 [00:00<?, ? examples/s]

...

Generating validation split:   0%|          | 0/4367 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4368 [00:00<?, ? examples/s]

Map:   0%|          | 0/55023 [00:00<?, ? examples/s]

Map:   0%|          | 0/4368 [00:00<?, ? examples/s]

IHC        — train: 19,332  test: 2,148
ISHate     — train: 55,023  test: 4,368
Vicomtech  — train: 1,914  test: 478


## 4. Self-Exclusion Lookup

`chunks_example_noihc.csv` maps raw tweet text → `chunk_id` in the FAISS index.
Used at train time to pass `chunk_id` so a model never retrieves itself as a neighbor.

In [4]:
from training_utils import compute_metrics, tokenize_augmented, strip_label_prefix, set_seed

chunks_df = pd.read_csv(CHUNKS_DIR / 'chunks_example_noihc.csv')

text_to_chunk_id = {
    strip_label_prefix(row.text): int(row.chunk_id)
    for _, row in chunks_df.iterrows()
}
print(f'Self-exclusion lookup: {len(text_to_chunk_id):,} entries')

Self-exclusion lookup: 49,049 entries


## 5. Augmentation Function

Retrieves k neighbors for each split. Called once per index inside the training loop.

In [5]:
def augment_split(hf_dataset, text_col, is_train, ret_model, ret_tokenizer, ret_index, ret_documents):
    records = []
    for example in tqdm(hf_dataset, desc=f"{'train' if is_train else 'test'}"):
        tweet    = example[text_col]
        chunk_id = text_to_chunk_id.get(tweet) if is_train else None
        neighbors = retrieve_top_k_above_threshold(
            tweet, THRESHOLD, ret_model, ret_tokenizer, ret_index, ret_documents,
            chunk_id=chunk_id, k=K, use_mean_pool=True,
        )
        records.append({
            'query':     tweet,
            'neighbors': [text for text, _ in neighbors],
            'label':     example['label'],
        })
    return records

## 6. Tokenization

Assembles the augmented string using the model's `sep_token` then tokenizes.

In [6]:
# tokenize_augmented imported from training_utils above

## 7. Metrics

In [7]:
# compute_metrics imported from training_utils above

## 8. Training Loop

For each (model, index_type, train_dataset): augment, fine-tune, evaluate on the IHC test set, and save weights.

In [ ]:
results = {}

print(f"Loading retriever: {RETRIEVER_HF_ID} ...")
ret_tokenizer = AutoTokenizer.from_pretrained(RETRIEVER_HF_ID)
ret_model     = AutoModel.from_pretrained(RETRIEVER_HF_ID).eval().to(device)
print(f"Retriever ready on {device}\n")

for index_type in SELECTED_INDEX_TYPES:
    index_path = INDEX_DIR / f'vdb_{index_type}.faiss'
    ret_index  = faiss.read_index(str(index_path))
    print(f"\n{'#'*60}")
    print(f"# Index: {index_type}  |  Vectors: {ret_index.ntotal:,}")
    print(f"{'#'*60}")

    with open(INDEX_DIR / f'lookup_{index_type}.json') as f:
        ret_documents = json.load(f)

    # ── Augment all train datasets + the fixed eval dataset ─────────────────
    aug_data = {}
    for ds_name in TRAIN_DATASETS + [EVAL_DATASET]:
        if ds_name in aug_data:
            continue
        ds_cfg = DATASETS[ds_name]
        print(f'\n=== Augmenting {ds_name} ===')
        aug_data[ds_name] = {
            'train': augment_split(ds_cfg['train'], ds_cfg['text_col'], True,
                                   ret_model, ret_tokenizer, ret_index, ret_documents),
            'test':  augment_split(ds_cfg['test'],  ds_cfg['text_col'], False,
                                   ret_model, ret_tokenizer, ret_index, ret_documents),
        }

    # ── Train on each TRAIN_DATASET, always evaluate on EVAL_DATASET ────────
    for model_name, hf_id in MODELS.items():
        if model_name not in SELECTED_MODELS:
            continue

        for train_ds in TRAIN_DATASETS:
            key = (model_name, index_type, train_ds)
            print(f"\n{'='*60}")
            print(f"Model: {model_name}  |  Index: {index_type}  |  Train: {train_ds}  |  Eval: {EVAL_DATASET}")
            print(f"{'='*60}")

            tokenizer = AutoTokenizer.from_pretrained(hf_id)
            tok_train = tokenize_augmented(aug_data[train_ds]['train'], tokenizer)
            tok_test  = tokenize_augmented(aug_data[EVAL_DATASET]['test'], tokenizer)

            # Seed before model init for reproducible classifier head initialization.
            # TrainingArguments(seed=42) only seeds the training loop, not from_pretrained().
            set_seed(42)
            model = AutoModelForSequenceClassification.from_pretrained(hf_id, num_labels=2)

            save_path = str(WEIGHTS_RAC_DIR / model_name / 'sbert' / index_type / train_ds)
            os.makedirs(save_path, exist_ok=True)

            training_args = TrainingArguments(
                output_dir=str(Path('../..') / 'checkpoints_rac_cross_evaluation' / model_name / 'sbert' / index_type / train_ds),
                num_train_epochs=NUM_EPOCHS,
                per_device_train_batch_size=BATCH_SIZE,
                per_device_eval_batch_size=BATCH_SIZE * 2,
                learning_rate=LEARNING_RATE,
                eval_strategy='epoch',
                save_strategy='no',
                logging_strategy='epoch',
                report_to='none',
                seed=42,
            )

            trainer = Trainer(
                model=model,
                args=training_args,
                train_dataset=tok_train,
                eval_dataset=tok_test,
                compute_metrics=compute_metrics,
            )

            trainer.train()
            trainer.save_model(save_path)
            tokenizer.save_pretrained(save_path)
            print(f'  Weights saved → {save_path}')

            preds_out = trainer.predict(tok_test)
            preds  = np.argmax(preds_out.predictions, axis=-1)
            labels = [r['label'] for r in aug_data[EVAL_DATASET]['test']]
            print(classification_report(labels, preds, target_names=['Non-HS', 'HS']))

            results[key] = {
                'macro_f1': f1_score(labels, preds, average='macro',  zero_division=0),
                'macro_p':  precision_score(labels, preds, average='macro', zero_division=0),
                'macro_r':  recall_score(labels, preds, average='macro',    zero_division=0),
            }

            del model
            if device.type == 'cuda':
                torch.cuda.empty_cache()

del ret_model, ret_tokenizer
if device.type == 'cuda':
    torch.cuda.empty_cache()

## 9. Results

Macro F1/P/R for each (model, index_type, train_dataset) combination.

In [9]:
metric_labels = {'macro_f1': 'F1', 'macro_p': 'Precision', 'macro_r': 'Recall'}

rows = {}
for (m, it, train_ds), vals in results.items():
    row_key = f"{m}/sbert/{it}/{train_ds}"
    if row_key not in rows:
        rows[row_key] = {}
    for metric, label in metric_labels.items():
        rows[row_key][(EVAL_DATASET, label)] = vals[metric]

df = pd.DataFrame(rows).T
df.columns = pd.MultiIndex.from_tuples(df.columns)
df.index.name = 'Model / Retriever / Index / TrainDataset'

styled = (
    df.style
    .format('{:.3f}')
    .highlight_max(axis=0, props='font-weight: bold; background-color: #d4f1d4')
    .set_caption(f'Cross-dataset RAG results — trained on ISHate/Vicomtech, evaluated on {EVAL_DATASET}')
)
display(styled)